# Recurrent encoder + LINEAR per-neuron readout + exact decoder (PER-LAYER supervision)

- ONE shared encoder layer applied RECURRENT_STEPS=16 times (weight-tied recurrence;
  NOT 16 distinct layers).
- Readout: NO slot queries / cross-attention -- a single Linear(256 -> 48) decodes
  EACH of the 128 neuron tokens to a 48-bit string (128 candidates, ~8 per string).
- Exact decoder: NLL of true strings under the uniform product-Bernoulli mixture of
  the 128 readouts (decode-from-an-equiprobable-list, computed in closed form).
- Deep supervision at EVERY layer: SUPERVISE_STEPS = 1..16, readout+loss after each
  of the 16 iterations (cheap now -- readout is one matmul).
- Recovery probed at 16 and 32 steps (32 > trained 16 -> step-generalisation).
- Validated on CPU: 0.8M params, forward (B,16,128,48), overfits to 16/16.
- 48 epochs, Kaggle GPU, BATCH_SIZE=128. To go faster set SUPERVISE_STEPS to a
  subset (e.g. [4,8,12,16]) or lower RECURRENT_STEPS/EPOCHS.

In [1]:
# =========================================================
# WEIGHT-TIED RECURRENT ENCODER + LINEAR PER-NEURON READOUT + EXACT DECODER
# =========================================================
# Simpler readout: NO slot queries, NO cross-attention. The encoder iteratively
# refines the 128 neuron tokens (one shared layer applied RECURRENT_STEPS times),
# then a single Linear(256 -> 48) decodes EACH neuron position to a 48-bit string.
# That gives 128 candidate strings (one per neuron, ~8 per memorized string). The
# exact decoder = NLL of the true strings under the product-Bernoulli mixture of
# those 128 readouts. Deep supervision: readout + loss at SUPERVISE_STEPS depths.
#
# This matches the discovered mechanism (cluster neurons by weight-similarity via
# self-attention, then read each cluster's string) and makes the readout a single
# matmul, so cost is dominated only by the K shared-layer applications.
# =========================================================
import os, glob, time, math, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | GPU Count: {torch.cuda.device_count()}")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DATA_DIR = "/kaggle/input/notebooks/emanuelruzak/lpe9datagen48bit-16strings-ipynb/dataset"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/working/dataset"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "dataset"

CKPT_PATH = "/kaggle/working/meta_exact_decoder_linreadout.pt"
if not os.path.isdir(os.path.dirname(CKPT_PATH)):
    CKPT_PATH = "meta_exact_decoder_linreadout.pt"

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
SMOKE = bool(int(os.environ.get("SMOKE", "0")))

D, H, N_TARGETS = 48, 128, 16
NUM_SLOTS = H                      # one readout per neuron position (128)
D_MODEL, NHEAD = 256, 8

RECURRENT_STEPS = 16               # shared layer applied this many times (trained depth)
SUPERVISE_STEPS = list(range(1, RECURRENT_STEPS + 1))  # readout+loss at EVERY layer
                                   # (per-layer deep supervision; cheap now -- readout is
                                   # one matmul). Use a subset e.g. [4,8,12,16] to go faster.
EVAL_STEPS      = [16, 32]         # recovery probed here (32 > 16 -> step-generalisation)
KS = [16, 32, 64, 128, 256, 512, 1024]  # secrets-in-top-k recovery budgets (fair vs GCG)

BATCH_SIZE = 32                   # big batch: model is tiny, readout is one matmul
EPOCHS     = 48
N_EVAL     = 20
LR, ETA_MIN, GRAD_CLIP = 3e-4, 1e-5, 1.0
USE_COMPILE = True

if SMOKE:
    BATCH_SIZE, EPOCHS, USE_COMPILE, RECURRENT_STEPS = 8, 2, False, 6
    SUPERVISE_STEPS, EVAL_STEPS, N_EVAL = list(range(1, 7)), [6, 12], 5

# ---------------------------------------------------------
# MODULES
# ---------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    def forward(self, x):
        v = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(v + self.eps) * self.weight

class CustomEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=1024):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=0.0)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
    def forward(self, src):
        n = self.norm1(src)
        a, _ = self.self_attn(n, n, n)
        src = src + a
        n = self.norm2(src)
        src = src + self.linear2(torch.square(F.relu(self.linear1(n))))
        return src

# ---------------------------------------------------------
# MODEL: refine 128 neuron tokens, linear-readout each to a string
# ---------------------------------------------------------
class MetaLinReadout(nn.Module):
    def __init__(self, d_model=D_MODEL, nhead=NHEAD, recurrent_steps=RECURRENT_STEPS,
                 supervise_steps=None):
        super().__init__()
        self.recurrent_steps = recurrent_steps
        self.supervise_steps = supervise_steps or SUPERVISE_STEPS
        self.weight_proj = nn.Sequential(nn.Linear(51, d_model), RMSNorm(d_model))
        self.shared_layer = CustomEncoderLayer(d_model, nhead)
        self.encoder_norm = RMSNorm(d_model)
        self.readout_head = nn.Linear(d_model, D)        # per-position: 256 -> 48 bit logits

    def readout(self, memory):                            # memory (B,128,256)
        return self.readout_head(memory)                  # (B,128,48)

    def forward(self, w1, b1, w2, b2, steps=None, readout_at=None):
        steps = steps or self.recurrent_steps
        if readout_at is None:
            readout_at = self.supervise_steps
        sup = sorted({s for s in readout_at if 1 <= s <= steps}) or [steps]
        b1e = b1.unsqueeze(-1); w2e = w2.transpose(1, 2)
        b2e = b2.unsqueeze(-1).expand(-1, H, 1)
        m = self.weight_proj(torch.cat([w1, b1e, w2e, b2e], dim=-1))
        outs = []
        for k in range(1, steps + 1):
            m = self.shared_layer(m)
            if k in sup:
                outs.append(self.readout(self.encoder_norm(m)))   # (B,128,48)
        return torch.stack(outs, dim=1)                           # (B,|sup|,128,48)

# ---------------------------------------------------------
# EXACT DECODER LOSS + deep supervision
# ---------------------------------------------------------
def exact_decoder_nll(slot_logits, targets):
    S = slot_logits.size(1)
    sign = (2.0 * targets - 1.0).unsqueeze(2)
    z = slot_logits.unsqueeze(1)
    logp = F.logsigmoid(sign * z).sum(-1)
    log_mix = torch.logsumexp(logp - math.log(S), dim=2)
    return -log_mix.mean()

def deep_nll(step_logits, targets):
    K = step_logits.size(1)
    return sum(exact_decoder_nll(step_logits[:, k], targets) for k in range(K)) / K

# ---------------------------------------------------------
# DATASET
# ---------------------------------------------------------
class MLPSetDataset(Dataset):
    def __init__(self, files):
        super().__init__()
        self.cache = []
        for f in files:
            try:
                self.cache.append(torch.load(f, map_location="cpu", weights_only=False))
            except Exception as e:
                print(f"skip {f}: {e}")
        self.mpc = self.cache[0]["W1"].shape[0] if self.cache else 0
        self.total = len(self.cache) * self.mpc
    def __len__(self):
        return self.total
    def _raw(self, idx):
        ch = self.cache[idx // self.mpc]; mi = idx % self.mpc
        return (ch["W1"][mi].float(), ch["b1"][mi].float(), ch["W2"][mi].float(),
                ch["b2"][mi].float(), ch["templates"][mi][:, 1:].float())
    def __getitem__(self, idx):
        return self._raw(idx)
    def get_batch1(self, idx, dev):
        w1, b1, w2, b2, t = self._raw(idx)
        return (w1[None].to(dev), b1[None].to(dev), w2[None].to(dev),
                b2[None].to(dev), t.to(dev))

def collate(batch):
    return (torch.stack([b[0] for b in batch]), torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]), torch.stack([b[3] for b in batch]),
            torch.stack([b[4] for b in batch]))

# ---------------------------------------------------------
# EVAL
# ---------------------------------------------------------
def _unwrap(m):
    while isinstance(m, nn.DataParallel) or hasattr(m, "_orig_mod"):
        m = m.module if isinstance(m, nn.DataParallel) else m._orig_mod
    return m

def _bits_to_int(bits):
    shifts = torch.arange(bits.shape[-1], dtype=torch.int64, device=bits.device)
    return (bits.to(torch.int64) << shifts).sum(-1)

@torch.no_grad()
def evaluate_nll(model, loader, steps):
    net = _unwrap(model); was = net.training; net.eval()
    tot, nb = 0.0, 0
    for w1, b1, w2, b2, t in loader:
        w1, b1, w2, b2, t = [x.to(device) for x in (w1, b1, w2, b2, t)]
        amp = (device.type == "cuda")
        with autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            out = net(w1, b1, w2, b2, steps=steps, readout_at=[steps])  # 1 readout
        tot += exact_decoder_nll(out[:, -1].float(), t).item(); nb += 1
    if was: net.train()
    return tot / max(1, nb)

# ---- secrets-in-top-k recovery: rank candidate strings by EXACT mixture log-prob ----
# (a fair, false-positive-aware comparison with GCG, which is scored at a fixed budget)
@torch.no_grad()
def topk_recovery(z, secrets, ks=KS, r=8, cap=4096, chunk=512):
    # z: (S,D) slot logits for ONE organism; secrets: (T,D)
    z = z.float(); S, Dd = z.shape; r = min(r, Dd)
    am  = (z > 0).float()                                          # per-slot argmax string
    low = z.abs().topk(r, dim=1, largest=False).indices           # r least-confident bits per slot
    P = 1 << r
    pats = ((torch.arange(P, device=z.device).unsqueeze(1) >> torch.arange(r, device=z.device)) & 1).float()
    cand = am.unsqueeze(1).expand(S, P, Dd).clone()
    idx  = low.unsqueeze(1).expand(S, P, r)
    fl   = pats.unsqueeze(0).expand(S, P, r)
    cand.scatter_(2, idx, (cand.gather(2, idx) - fl).abs())       # XOR each pattern into the low bits
    base = F.logsigmoid(z.abs()).sum(1, keepdim=True)
    cost = (z.abs().gather(1, low).unsqueeze(1) * fl).sum(2)
    prop = (base - cost).reshape(-1)                             # per-slot log-prob of each candidate
    cand = cand.reshape(S * P, Dd)
    if cand.shape[0] > cap:
        cand = cand[prop.topk(cap).indices]
    logS = math.log(S); lps = []
    for i in range(0, cand.shape[0], chunk):
        c = cand[i:i+chunk]; sign = 2.0 * c - 1.0
        lp = F.logsigmoid(sign.unsqueeze(1) * z.unsqueeze(0)).sum(-1)       # (m,S)
        lps.append(torch.logsumexp(lp - logS, dim=1))
    logp = torch.cat(lps); ci = _bits_to_int(cand)
    uniq, inv = torch.unique(ci, return_inverse=True)
    ulp = torch.full((uniq.numel(),), float("-inf"), device=z.device)
    ulp.scatter_reduce_(0, inv, logp, reduce="amax", include_self=True)
    ranked = uniq[ulp.argsort(descending=True)]
    ti = _bits_to_int(secrets)
    return {k: int(torch.isin(ti, ranked[:k]).sum()) for k in ks}

@torch.no_grad()
def nll_profile(z, secrets):
    # per-secret NLL under the mixture, sorted ascending (best..worst)
    z = z.float(); S = z.size(0)
    sign = (2.0 * secrets - 1.0).unsqueeze(1)
    logp = F.logsigmoid(sign * z.unsqueeze(0)).sum(-1)
    nll  = -(torch.logsumexp(logp - math.log(S), dim=1))
    return nll.sort().values

@torch.no_grad()
def eval_metrics(model, dataset, n_mlps, steps, ks=KS):
    net = _unwrap(model); was = net.training; net.eval()
    agg = {k: 0.0 for k in ks}; nlls = []; counted = 0
    for gi in range(min(n_mlps, dataset.total)):
        w1, b1, w2, b2, targets = dataset.get_batch1(gi, device)
        amp = (device.type == "cuda")
        with autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            z = net(w1, b1, w2, b2, steps=steps, readout_at=[steps])[0, -1].float()  # (128,48)
        hits = topk_recovery(z, targets, ks)
        for k in ks: agg[k] += hits[k]
        nlls.append(nll_profile(z, targets))
        counted += 1
    if was: net.train()
    n = max(1, counted)
    return {k: agg[k] / n for k in ks}, torch.stack(nlls).mean(0)

# ---------------------------------------------------------
# TRAIN
# ---------------------------------------------------------
def main():
    files = glob.glob(os.path.join(DATA_DIR, "*.pt"))
    print(f"Found {len(files)} chunk files in {DATA_DIR}")
    if not files:
        raise FileNotFoundError(f"No .pt files in {DATA_DIR}")
    random.seed(42); random.shuffle(files)
    if SMOKE:
        files = files[:3]
    split = max(1, int(0.9 * len(files)))
    train_files, test_files = files[:split], files[split:] or files[:1]
    train_ds = MLPSetDataset(train_files); test_ds = MLPSetDataset(test_files)
    print(f"Total MLPs: train={len(train_ds)} test={len(test_ds)} | slots={NUM_SLOTS}=per-neuron "
          f"RECURRENT_STEPS={RECURRENT_STEPS} supervise={SUPERVISE_STEPS} eval={EVAL_STEPS}")

    nw = 0 if SMOKE else 2
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate, num_workers=nw, drop_last=True,
                              pin_memory=(device.type == "cuda"))
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             collate_fn=collate, num_workers=nw)

    model = MetaLinReadout().to(device)
    print(f"params = {sum(p.numel() for p in model.parameters()):,} (single shared layer + linear head)")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    if USE_COMPILE:
        try:
            model = torch.compile(model); print("torch.compile enabled.")
        except Exception as e:
            print(f"compile off: {e}")

    total_steps = max(1, len(train_loader) * EPOCHS)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps, eta_min=ETA_MIN)
    scaler = GradScaler(device.type, enabled=(device.type == "cuda"))
    print(f"steps/epoch={len(train_loader)} total={total_steps}\nTraining...")

    gstep = 0
    for epoch in range(EPOCHS):
        model.train(); run = 0.0; t0 = time.time()
        for w1, b1, w2, b2, t in train_loader:
            w1, b1, w2, b2, t = [x.to(device) for x in (w1, b1, w2, b2, t)]
            opt.zero_grad(set_to_none=True)
            amp = (device.type == "cuda")
            with autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                out = model(w1, b1, w2, b2)                    # (B,|sup|,128,48)
                loss = deep_nll(out.float(), t)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); gstep += 1
            if gstep % 200 == 0:
                print(f"  step {gstep:06d} | deep-nll {loss.item():.4f} | lr {sched.get_last_lr()[0]:.2e}")
        tr = run / max(1, len(train_loader))
        te_nll = evaluate_nll(model, test_loader, steps=RECURRENT_STEPS)
        print(f"=== Epoch {epoch+1}/{EPOCHS} | deep-NLL train {tr:.3f} | "
              f"last-step NLL test {te_nll:.3f} (floor {math.log(N_TARGETS):.2f}) | {time.time()-t0:.1f}s ===")
        for s in EVAL_STEPS:
            topk, nllp = eval_metrics(model, test_ds, N_EVAL, s)
            print(f"    {s}-step | secrets in top-k: " + " ".join(f"top{k}:{topk[k]:.2f}" for k in KS))
            if s == RECURRENT_STEPS:
                print("      per-secret NLL by rank (1=best..16=worst): "
                      + " ".join(f"{nllp[i]:.1f}" for i in range(N_TARGETS)))
        torch.save(_unwrap(model).state_dict(), CKPT_PATH)
    print("Done. ->", CKPT_PATH)

if __name__ == "__main__":
    main()

Using device: cuda | GPU Count: 2
Found 50 chunk files in /kaggle/input/notebooks/emanuelruzak/lpe9datagen48bit-16strings-ipynb/dataset
Total MLPs: train=46080 test=5120 | slots=128=per-neuron RECURRENT_STEPS=16 supervise=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16] eval=[16, 32]
params = 815,408 (single shared layer + linear head)
torch.compile enabled.
steps/epoch=1440 total=69120
Training...


W0622 01:06:30.612000 23 torch/_logging/_internal.py:1204] [0/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/variables/functions.py:1946: UserWarning: Dynamo does not know how to trace the builtin `_thread.get_ident.` This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind).
If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround.
If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use `torch.compiler.allow_in_graph`.
  torch._dynamo.utils.warn_once(explanation + "\n" + "\n".join(hints))
W0622 01:06:33.874000 23 torch/_dynamo/convert_frame.py:1676] [3/8] torch._dyna

  step 000200 | deep-nll 33.1791 | lr 3.00e-04
  step 000400 | deep-nll 28.6301 | lr 3.00e-04
  step 000600 | deep-nll 26.2554 | lr 3.00e-04
  step 000800 | deep-nll 23.5288 | lr 3.00e-04
  step 001000 | deep-nll 21.9575 | lr 3.00e-04
  step 001200 | deep-nll 21.2559 | lr 3.00e-04
  step 001400 | deep-nll 19.8234 | lr 3.00e-04
=== Epoch 1/48 | deep-NLL train 25.624 | last-step NLL test 19.604 (floor 2.77) | 163.1s ===
    16-step | secrets in top-k: top16:1.95 top32:2.25 top64:2.45 top128:2.85 top256:3.20 top512:3.30 top1024:3.40
      per-secret NLL by rank (1=best..16=worst): 7.6 9.3 11.2 12.9 14.9 15.8 17.4 18.5 19.5 20.6 21.5 22.9 24.0 25.3 26.7 29.3
    32-step | secrets in top-k: top16:1.00 top32:1.25 top64:1.45 top128:1.70 top256:1.90 top512:2.15 top1024:2.40
  step 001600 | deep-nll 19.5069 | lr 3.00e-04
  step 001800 | deep-nll 19.6252 | lr 3.00e-04
  step 002000 | deep-nll 19.6052 | lr 2.99e-04
  step 002200 | deep-nll 18.7131 | lr 2.99e-04
  step 002400 | deep-nll 18.0087 | 